Persiapan Data & Text Preprocessing

In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import f1_score, classification_report

# Download stopwords dari NLTK
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# 1. Load Data (Menggunakan raw link publik agar praktis)
url = 'https://raw.githubusercontent.com/laxmimerit/twitter-disaster-prediction-dataset/master/train.csv'
df = pd.read_csv(url)

# 2. Text Preprocessing Function
def clean_text(text):
    text = text.lower() # Lowercasing
    text = re.sub(r'http\S+|www\.\S+', '', text) # Hapus URL
    text = re.sub(r'@\w+', '', text) # Hapus Mention
    text = re.sub(r'[^a-zA-Z\s]', '', text) # Hapus Tanda Baca/Karakter Spesial
    # Hapus Stopwords (opsional, tapi bagus untuk TF-IDF)
    text = " ".join([word for word in text.split() if word not in stop_words])
    return text

# Aplikasikan fungsi pembersihan
df['clean_text'] = df['text'].apply(clean_text)

# 3. Train-Validation Split
X_train, X_val, y_train, y_val = train_test_split(df['clean_text'], df['target'], test_size=0.2, random_state=42)

print("Preprocessing Selesai! Contoh teks bersih:")
print(X_train.head(3))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Preprocessing Selesai! Contoh teks bersih:
4996    courageous honest analysis need use atomic bom...
3263    wld b shame golf cart became engulfed flames b...
4907    tell rescind medals honor given us soldiers ma...
Name: clean_text, dtype: object


Baseline Model Konvensional (TF-IDF + LR & TF-IDF + NB)

In [ ]:
# Ekstraksi Fitur: TF-IDF (Unigram & Bigram)
tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)

# --- Kombinasi 1: TF-IDF + Logistic Regression ---
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_tfidf, y_train)
lr_pred = lr_model.predict(X_val_tfidf)
lr_f1 = f1_score(y_val, lr_pred)

# --- Kombinasi 2: TF-IDF + Multinomial Naive Bayes ---
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
nb_pred = nb_model.predict(X_val_tfidf)
nb_f1 = f1_score(y_val, nb_pred)

print(f"=== Baseline F1-Score ===")
print(f"[TF-IDF + Logistic Regression] F1-Score: {lr_f1:.4f}")
print(f"[TF-IDF + Naive Bayes]         F1-Score: {nb_f1:.4f}")

=== Baseline F1-Score ===
[TF-IDF + Logistic Regression] F1-Score: 0.7531
[TF-IDF + Naive Bayes]         F1-Score: 0.7400


Model Deep Learning (Word Embedding + BiLSTM)

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dense, Dropout

# Parameter Tokenizer & Padding
MAX_WORDS = 10000
MAX_LEN = 50

# Tokenisasi & Padding (Mengubah kata menjadi urutan angka integer)
tokenizer = Tokenizer(num_words=MAX_WORDS)
tokenizer.fit_on_texts(X_train)

X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=MAX_LEN)
X_val_seq = pad_sequences(tokenizer.texts_to_sequences(X_val), maxlen=MAX_LEN)

# Membangun Arsitektur BiLSTM
dl_model = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=128, input_length=MAX_LEN),
    Bidirectional(LSTM(64, return_sequences=False)),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

dl_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Custom F1-Score Metrik (karena Keras tidak memiliki metrik F1-Score bawaan)
print("Melatih Model Deep Learning (BiLSTM)...")
history = dl_model.fit(X_train_seq, y_train, epochs=4, batch_size=64, validation_data=(X_val_seq, y_val), verbose=1)

# Prediksi & Evaluasi F1-Score DL
dl_prob = dl_model.predict(X_val_seq)
dl_pred = (dl_prob > 0.5).astype(int).reshape(-1)
dl_f1 = f1_score(y_val, dl_pred)

print(f"\n[Deep Learning - BiLSTM] F1-Score: {dl_f1:.4f}")

Melatih Model Deep Learning (BiLSTM)...
Epoch 1/4


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


96/96 ━━━━━━━━━━━━━━━━━━━━ 18s 129ms/step - accuracy: 0.6399 - loss: 0.6296 - val_accuracy: 0.7814 - val_loss: 0.4811
Epoch 2/4
96/96 ━━━━━━━━━━━━━━━━━━━━ 12s 123ms/step - accuracy: 0.8557 - loss: 0.3713 - val_accuracy: 0.7912 - val_loss: 0.4780
Epoch 3/4
96/96 ━━━━━━━━━━━━━━━━━━━━ 12s 128ms/step - accuracy: 0.9176 - loss: 0.2365 - val_accuracy: 0.7768 - val_loss: 0.5551
Epoch 4/4
96/96 ━━━━━━━━━━━━━━━━━━━━ 12s 125ms/step - accuracy: 0.9461 - loss: 0.1683 - val_accuracy: 0.7531 - val_loss: 0.7173
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step

[Deep Learning - BiLSTM] F1-Score: 0.7130


Analisis Error & Laporan Akhir

In [ ]:
# Gabungkan hasil untuk analisis
val_df = pd.DataFrame({
    'text': df.loc[y_val.index, 'text'],
    'clean_text': X_val,
    'true_label': y_val,
    'lr_pred': lr_pred,
    'dl_pred': dl_pred
})

# Cari kasus di mana TF-IDF Salah, tapi Deep Learning Benar
dl_better = val_df[(val_df['lr_pred'] != val_df['true_label']) & (val_df['dl_pred'] == val_df['true_label'])]

print("Contoh Kasus di mana DL (BiLSTM) lebih pintar dari TF-IDF (Konvensional):")
for idx, row in dl_better.head(3).iterrows():
    label_str = "BENCANA NYATA (1)" if row['true_label'] == 1 else "BUKAN BENCANA (0)"
    print(f"\nTweet Asli: {row['text']}")
    print(f"Status Sebenarnya : {label_str}")
    print(f"Prediksi TF-IDF   : {row['lr_pred']} (SALAH)")
    print(f"Prediksi BiLSTM   : {row['dl_pred']} (BENAR)")

Contoh Kasus di mana DL (BiLSTM) lebih pintar dari TF-IDF (Konvensional):

Tweet Asli: @brianroemmele UX fail of EMV - people want to insert and remove quickly like a gas pump stripe reader. 1 person told me it crashed the POS
Status Sebenarnya : BENCANA NYATA (1)
Prediksi TF-IDF   : 0 (SALAH)
Prediksi BiLSTM   : 1 (BENAR)

Tweet Asli: 'The Reagan Administration had arranged for Israeli weapons to be sent to the Guatemalan Army  http://t.co/4fYNQ1hWWb
Status Sebenarnya : BENCANA NYATA (1)
Prediksi TF-IDF   : 0 (SALAH)
Prediksi BiLSTM   : 1 (BENAR)

Tweet Asli: Back to the future in #Vanuatu how Cyclone Pam has encouraged traditional ways of living:
http://t.co/aFMKcFn1TL http://t.co/6QZXFK2LFS
Status Sebenarnya : BENCANA NYATA (1)
Prediksi TF-IDF   : 0 (SALAH)
Prediksi BiLSTM   : 1 (BENAR)
